In [1]:
# 2. Librerías necesarias
#from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator



In [4]:
# 3. Subir archivo CSV
#uploaded = files.upload()

# 4. Crear sesión de Spark
spark = SparkSession.builder.appName("SkinCareML").getOrCreate()

# 5. Cargar datos en un DataFrame de Spark
df = spark.read.csv("skincare_products.csv", header=True, inferSchema=True)

# Mostrar primeras filas
print("📊 Primeras filas del dataset:")
df.show(5)

# Resumen estadístico
print("📈 Resumen estadístico:")
df.describe().show()
df


📊 Primeras filas del dataset:
+-----------------+-----------+---------+---+------------+
|     Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|
+-----------------+-----------+---------+---+------------+
|Ácido Hialurónico|       Alto|    Medio|  0|        Seco|
|          Retinol|       Bajo|     Alto|  0|       Graso|
|       Vitamina C|      Medio|    Medio| 30|       Mixto|
|        Aloe Vera|       Alto|     Bajo| 15|    Sensible|
|      Niacinamida|      Medio|    Medio|  0|       Mixto|
+-----------------+-----------+---------+---+------------+
only showing top 5 rows
📈 Resumen estadístico:
+-------+----------------+-----------+---------+-----------------+------------+
|summary|    Ingredientes|Hidratación|Absorción|              SPF|Tipo de Piel|
+-------+----------------+-----------+---------+-----------------+------------+
|  count|              20|         20|       20|               20|          20|
|   mean|            NULL|       NULL|     NULL|              7.5|      

DataFrame[Ingredientes: string, Hidratación: string, Absorción: string, SPF: int, Tipo de Piel: string]

In [5]:
# ============================================================
# 2. Preprocesamiento de Datos
# ============================================================

# Mapear "Tipo de Piel" a valores numéricos
df = df.withColumn("label",
    when(df["Tipo de Piel"] == "Seco", 0)
    .when(df["Tipo de Piel"] == "Graso", 1)
    .when(df["Tipo de Piel"] == "Mixto", 2)
    .when(df["Tipo de Piel"] == "Sensible", 3)
)
print("Se crea la columna label, que corresponde al tipo de piel")
df.show(5)
# Mapear Hidratación
df = df.withColumn("Hidratacion_num",
    when(df["Hidratación"] == "Bajo", 0)
    .when(df["Hidratación"] == "Medio", 1)
    .when(df["Hidratación"] == "Alto", 2)
)
print("Se crea la columna hidratacion_num, que corresponde a la calidad de la hidratacion de la piel")
df.show(5)
# Mapear Absorción
df = df.withColumn("Absorcion_num",
    when(df["Absorción"] == "Bajo", 0)
    .when(df["Absorción"] == "Medio", 1)
    .when(df["Absorción"] == "Alto", 2)
)
print("Se crea la columna absorción, que corresponde a la calidad de absorción de la piel")
df.show(5)

# Seleccionar columnas relevantes
feature_cols = ["Hidratacion_num", "Absorcion_num", "SPF"]

# Ensamblar vector de características
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df = assembler.transform(df)

print("✅ Datos preprocesados:")
df.select("Hidratación", "Absorción", "SPF", "Tipo de Piel", "label", "features").show(5)



Se crea la columna label, que corresponde al tipo de piel
+-----------------+-----------+---------+---+------------+-----+
|     Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|label|
+-----------------+-----------+---------+---+------------+-----+
|Ácido Hialurónico|       Alto|    Medio|  0|        Seco|    0|
|          Retinol|       Bajo|     Alto|  0|       Graso|    1|
|       Vitamina C|      Medio|    Medio| 30|       Mixto|    2|
|        Aloe Vera|       Alto|     Bajo| 15|    Sensible|    3|
|      Niacinamida|      Medio|    Medio|  0|       Mixto|    2|
+-----------------+-----------+---------+---+------------+-----+
only showing top 5 rows
Se crea la columna hidratacion_num, que corresponde a la calidad de la hidratacion de la piel
+-----------------+-----------+---------+---+------------+-----+---------------+
|     Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|label|Hidratacion_num|
+-----------------+-----------+---------+---+------------+-----+-------------

In [6]:
# ============================================================
# 3. División de datos y entrenamiento del modelo
# ============================================================

# Dividir dataset en 80% train, 20% test
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)
print("train data")
train_data.show()
print("test data")
test_data.show()
# Modelo de Árbol de Decisión
# Jugar con el hiperparámetro maxDepth (profundidad máxima)
# Por ejemplo, cambiamos maxDepth a 3 (originalmente no estaba especificado, usando el valor por defecto)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)


# Entrenar modelo
model = dt.fit(train_data)


train data
+-------------------+-----------+---------+---+------------+-----+---------------+-------------+--------------+
|       Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|label|Hidratacion_num|Absorcion_num|      features|
+-------------------+-----------+---------+---+------------+-----+---------------+-------------+--------------+
|          Aloe Vera|       Alto|     Bajo| 15|    Sensible|    3|              2|            0|[2.0,0.0,15.0]|
|           Arbutina|      Medio|     Alto|  0|       Mixto|    2|              1|            2| [1.0,2.0,0.0]|
|      Beta-Glucanos|       Alto|     Bajo| 10|    Sensible|    3|              2|            0|[2.0,0.0,10.0]|
|  Centella Asiática|      Medio|    Medio| 20|    Sensible|    3|              1|            1|[1.0,1.0,20.0]|
|          Ceramidas|       Alto|     Bajo|  0|        Seco|    0|              2|            0| [2.0,0.0,0.0]|
|Extracto de Regaliz|      Medio|    Medio| 15|    Sensible|    3|              1|           

25/09/08 22:32:35 WARN DecisionTreeMetadata: DecisionTree reducing maxBins from 32 to 15 (= number of training instances)


In [ ]:

# ============================================================
# 4. Predicción y Evaluación
# ============================================================

# Aplicar modelo al set de prueba
predictions = model.transform(test_data)

print("🔮 Predicciones realizadas:")
predictions.select("features", "label", "prediction").show(11)

# Evaluar precisión
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"✅ Precisión del modelo con maxDepth=3: {accuracy:.2f}")


🔮 Predicciones realizadas:
+-------------+-----+----------+
|     features|label|prediction|
+-------------+-----+----------+
|[0.0,1.0,0.0]|    2|       2.0|
|[2.0,1.0,0.0]|    0|       0.0|
|[1.0,2.0,0.0]|    2|       1.0|
|[0.0,2.0,0.0]|    1|       1.0|
|[0.0,2.0,0.0]|    1|       1.0|
+-------------+-----+----------+

✅ Precisión del modelo con maxDepth=3: 0.80


25/09/09 02:10:31 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 207542 ms exceeds timeout 120000 ms
25/09/09 02:10:31 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/09 02:10:31 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o

In [ ]:

# ============================================================
# 4. Predicción y Evaluación
# ============================================================

# Aplicar modelo al set de prueba
predictions = model.transform(test_data)

print("🔮 Predicciones realizadas:")
predictions.select("features", "label", "prediction").show(11)

# Evaluar precisión
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"✅ Precisión del modelo con maxDepth=3: {accuracy:.2f}")


🔮 Predicciones realizadas:
+-------------+-----+----------+
|     features|label|prediction|
+-------------+-----+----------+
|[0.0,1.0,0.0]|    2|       2.0|
|[2.0,1.0,0.0]|    0|       0.0|
|[1.0,2.0,0.0]|    2|       1.0|
|[0.0,2.0,0.0]|    1|       1.0|
|[0.0,2.0,0.0]|    1|       1.0|
+-------------+-----+----------+

✅ Precisión del modelo con maxDepth=3: 0.80


In [29]:

# ============================================================
# 5. Análisis de Resultados
# ============================================================
print ("""
Hemos probado el modelo de Árbol de Decisión con una profundidad máxima de 3.
Observa cómo la precisión cambia al variar este hiperparámetro. Una mayor
profundidad puede llevar a sobreajuste, mientras que una menor profundidad
puede simplificar demasiado el modelo.
""")


Hemos probado el modelo de Árbol de Decisión con una profundidad máxima de 3.
Observa cómo la precisión cambia al variar este hiperparámetro. Una mayor
profundidad puede llevar a sobreajuste, mientras que una menor profundidad
puede simplificar demasiado el modelo.



In [25]:
# ============================================================
# 🌲 Segundo Modelo: Random Forest Classifier
# ============================================================

from pyspark.ml.classification import RandomForestClassifier

# Crear modelo de Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=4)

# Entrenar modelo
rf_model = rf.fit(train_data)

# Realizar predicciones
rf_predictions = rf_model.transform(test_data)

print("🔮 Predicciones con Random Forest:")
rf_predictions.select("features", "label", "prediction").show(13)

# Evaluar precisión del modelo
rf_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
rf_accuracy = rf_evaluator.evaluate(rf_predictions)

print(f"✅ Precisión del modelo Random Forest: {rf_accuracy:.2f}")

# ============================================================
# 📊 Comparación de Modelos
# ============================================================

print("Comparación final:")
print(f"Árbol de Decisión -> Precisión: {accuracy:.2f}")
print(f"Random Forest     -> Precisión: {rf_accuracy:.2f}")

"""
El Random Forest suele mejorar la precisión porque combina múltiples árboles,
reduciendo el riesgo de sobreajuste y generalizando mejor en datasets pequeños como este.
"""

🔮 Predicciones con Random Forest:
+-------------+-----+----------+
|     features|label|prediction|
+-------------+-----+----------+
|[0.0,1.0,0.0]|    2|       2.0|
|[2.0,1.0,0.0]|    0|       0.0|
|[1.0,2.0,0.0]|    2|       1.0|
|[0.0,2.0,0.0]|    1|       1.0|
|[0.0,2.0,0.0]|    1|       1.0|
+-------------+-----+----------+

✅ Precisión del modelo Random Forest: 0.80
Comparación final:
Árbol de Decisión -> Precisión: 0.80
Random Forest     -> Precisión: 0.80


'\nEl Random Forest suele mejorar la precisión porque combina múltiples árboles,\nreduciendo el riesgo de sobreajuste y generalizando mejor en datasets pequeños como este.\n'

print("--- Inicio de la explicación de la sección 2 ---")

## 2. Preprocesamiento de Datos

En esta sección, se prepara el conjunto de datos para el entrenamiento del modelo.

*   **Mapeo de columnas categóricas a numéricas**: Las columnas "Tipo de Piel", "Hidratación" y "Absorción", que contienen valores de texto (categorías), se convierten a valores numéricos. Esto es necesario porque la mayoría de los algoritmos de Machine Learning trabajan con datos numéricos. Se utiliza la función `when` de PySpark para asignar un número a cada categoría.
*   **Selección de columnas relevantes**: Se definen las columnas que se utilizarán como características para entrenar el modelo. En este caso, son "Hidratacion_num", "Absorcion_num" y "SPF".
*   **Ensamblaje de vector de características**: Se crea un vector único que combina todas las columnas de características seleccionadas. Esto se hace utilizando `VectorAssembler`, que transforma las columnas individuales en un solo vector de características, que es el formato de entrada requerido por muchos modelos de Spark ML.

print("--- Fin de la explicación de la sección 2 ---")

print("--- Inicio de la explicación de la sección 3 ---")

## 3. División de datos y entrenamiento del modelo

Aquí se divide el dataset en conjuntos de entrenamiento y prueba, y se entrena un modelo de clasificación.

*   **División del dataset**: El DataFrame `df` se divide aleatoriamente en dos conjuntos: `train_data` (80% de los datos) y `test_data` (20% de los datos). Esto permite entrenar el modelo con una parte de los datos y evaluar su rendimiento con datos que no ha visto durante el entrenamiento. `seed=42` asegura que la división sea la misma cada vez que se ejecuta el código.
*   **Modelo de Árbol de Decisión**: Se inicializa un modelo de clasificación de Árbol de Decisión (`DecisionTreeClassifier`). Se especifican las columnas de características (`featuresCol="features"`) y la columna objetivo (`labelCol="label"`).
*   **Entrenamiento del modelo**: El modelo de Árbol de Decisión se entrena utilizando el conjunto de datos de entrenamiento (`train_data`) con el método `fit()`. Este proceso ajusta los parámetros del modelo para que pueda aprender a predecir el tipo de piel basándose en las características.

print("--- Fin de la explicación de la sección 3 ---")

print("--- Inicio de la explicación de la sección 4 ---")

## 4. Predicción y Evaluación

En esta sección, se utiliza el modelo entrenado para hacer predicciones en el conjunto de prueba y se evalúa su precisión.

*   **Aplicación del modelo al set de prueba**: El modelo entrenado (`model`) se aplica al conjunto de datos de prueba (`test_data`) utilizando el método `transform()`. Esto genera un nuevo DataFrame (`predictions`) que incluye una columna adicional con las predicciones del modelo (`prediction`).
*   **Evaluación de precisión**: Se utiliza un evaluador de clasificación multiclase (`MulticlassClassificationEvaluator`) para calcular la precisión del modelo. Se especifican las columnas de etiqueta real (`labelCol="label"`) y las columnas de predicción del modelo (`predictionCol="prediction"`) y la métrica a calcular (`metricName="accuracy"`). El método `evaluate()` calcula el valor de la métrica especificada.

print("--- Fin de la explicación de la sección 4 ---")

print("--- Inicio de la explicación de la sección 5 ---")

## 5. Análisis de Resultados

Esta sección contiene un comentario explicando los resultados obtenidos y posibles mejoras.

*   El comentario resume la precisión obtenida por el modelo.
*   Sugiere que la precisión indica que el modelo puede clasificar la mayoría de los productos correctamente.
*   Propone probar otros algoritmos de Machine Learning más avanzados (como Random Forest o Gradient Boosted Trees) y añadir más variables al dataset para intentar mejorar el rendimiento del modelo.

print("--- Fin de la explicación de la sección 5 ---")